In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class HamburgMambaConfig:
    def __init__(self):
        self.d_model = 512
        self.n_layers = 12
        self.d_state = 64
        self.d_conv = 4
        self.expand = 2
        self.dt_rank = "auto"
        self.d_inner = None
        self.bias = False
        self.conv_bias = True
        self.dropout = 0.1

        self.max_sequence_length = 2048
        self.safety_focus = True
        self.crisis_detection = True
        self.bidirectional = True

        self.num_hospitals = 6
        self.num_crisis_levels = 5
        self.num_acuity_levels = 5

        if self.d_inner is None:
            self.d_inner = int(self.expand * self.d_model)
        if self.dt_rank == "auto":
            self.dt_rank = math.ceil(self.d_model / 16)

class HamburgClinicalEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.vital_encoder = nn.Linear(7, 64)
        self.demographic_encoder = nn.Linear(2, 32)

        self.hospital_embedding = nn.Embedding(config.num_hospitals + 1, 32)
        self.acuity_embedding = nn.Embedding(config.num_acuity_levels + 1, 32)
        self.crisis_embedding = nn.Embedding(config.num_crisis_levels + 1, 32)

        self.crew_state_encoder = nn.Linear(4, 64)
        self.system_stress_encoder = nn.Linear(3, 32)

        self.safety_encoder = nn.Linear(3, 32)
        self.time_encoder = nn.Linear(3, 32)
        self.environment_encoder = nn.Linear(2, 32)

        self.complaint_embedding = nn.Embedding(20, 32)

        input_dim = 64 + 32 + 32 + 32 + 32 + 64 + 32 + 32 + 32 + 32 + 32
        self.final_projection = nn.Linear(input_dim, config.d_model)

    def forward(self, patient_data):
        vitals = torch.stack([
            patient_data['vital_heart_rate'],
            patient_data['vital_bp_systolic'],
            patient_data['vital_bp_diastolic'],
            patient_data['vital_respiratory_rate'],
            patient_data['vital_oxygen_saturation'],
            patient_data['vital_temperature'],
            patient_data['vital_gcs']
        ], dim=-1)
        vital_features = self.vital_encoder(vitals)

        demographics = torch.stack([
            patient_data['age'] / 100.0,
            patient_data['gender']
        ], dim=-1)
        demo_features = self.demographic_encoder(demographics)

        hospital_features = self.hospital_embedding(patient_data['hospital_destination'])
        acuity_features = self.acuity_embedding(patient_data['acuity_level'])
        crisis_features = self.crisis_embedding(patient_data['system_crisis_level'])

        crew_state = torch.stack([
            patient_data['crew_calls_today'] / 25.0,
            patient_data['crew_hours_on_shift'] / 12.0,
            patient_data['crew_fatigue_level'] / 2.0,
            patient_data['burnout_risk_score']
        ], dim=-1)
        crew_features = self.crew_state_encoder(crew_state)

        system_stress = torch.stack([
            patient_data['available_crews'] / 80.0,
            patient_data['calls_waiting'] / 10.0,
            patient_data['response_delay_minutes'] / 30.0
        ], dim=-1)
        stress_features = self.system_stress_encoder(system_stress)

        safety_indicators = torch.stack([
            patient_data['response_delay_minutes'] / 30.0,
            patient_data['handoff_quality_score'],
            patient_data['documentation_completeness']
        ], dim=-1)
        safety_features = self.safety_encoder(safety_indicators)

        temporal = torch.stack([
            patient_data['hour'] / 24.0,
            patient_data['day_of_week'] / 7.0,
            patient_data['shift_change_stress']
        ], dim=-1)
        time_features = self.time_encoder(temporal)

        environment = torch.stack([
            patient_data['weather_impact'],
            patient_data['tourism_factor']
        ], dim=-1)
        env_features = self.environment_encoder(environment)

        complaint_features = self.complaint_embedding(patient_data['chief_complaint_encoded'])

        all_features = torch.cat([
            vital_features,
            demo_features,
            hospital_features,
            acuity_features,
            crisis_features,
            crew_features,
            stress_features,
            safety_features,
            time_features,
            env_features,
            complaint_features
        ], dim=-1)

        return self.final_projection(all_features)

class HybridSafetyModel(nn.Module):
    def __init__(self, config, backbone_type="gru"):
        super().__init__()
        self.config = config
        self.backbone_type = backbone_type.lower()

        # Plug in HamburgClinicalEncoder
        self.encoder = HamburgClinicalEncoder(config)

        # Backbone: GRU or Transformer
        if self.backbone_type == "gru":
            self.backbone = nn.GRU(
                input_size=config.d_model,
                hidden_size=config.d_model,
                batch_first=True,
                bidirectional=False
            )
        elif self.backbone_type == "transformer":
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=config.d_model,
                nhead=4,
                dim_feedforward=4 * config.d_model,
                dropout=config.dropout,
                batch_first=True
            )
            self.backbone = nn.TransformerEncoder(encoder_layer, num_layers=2)
        else:
            raise ValueError("Unsupported backbone type")

        # Output heads
        self.crisis_head = nn.Sequential(
            nn.Linear(config.d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, config.num_crisis_levels)
        )

        self.staff_head = nn.Sequential(
            nn.Linear(config.d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 3)  # burnout, fatigue, retention
        )

        self.safety_head = nn.Sequential(
            nn.Linear(config.d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 3)  # delay, quality, adverse event risk
        )

    def forward(self, patient_data):
        x = self.encoder(patient_data)  # (B, L, d_model)

        if self.backbone_type == "gru":
            x, _ = self.backbone(x)  # GRU returns output, hidden
        elif self.backbone_type == "transformer":
            x = self.backbone(x)

        # Use last time step for predictions
        last_state = x[:, -1, :]

        crisis_logits = self.crisis_head(last_state)
        staff_output = self.staff_head(last_state)
        safety_output = self.safety_head(last_state)

        return {
            "crisis_logits": crisis_logits,
            "burnout": torch.sigmoid(staff_output[:, 0]),
            "fatigue": torch.sigmoid(staff_output[:, 1]) * 2.0,
            "retention": torch.sigmoid(staff_output[:, 2]),
            "response_delay": F.relu(safety_output[:, 0]),
            "quality": torch.sigmoid(safety_output[:, 1]),
            "adverse_risk": torch.sigmoid(safety_output[:, 2])
        }
